# SiDroForest — final methodological analysis and manuscript figures (v16)

This notebook consolidates the analyses needed for the revised manuscript. The previously developed Random Forest is retained as the **pixel-level classifier**; the new analysis focuses on independent evaluation, spatial forest composition, parameter sensitivity, uncertainty, and scale-aware product comparison.

## Primary analysis choices

- `0` = masked / uncertain
- `1` = evergreen needleleaf
- `2` = summergreen needleleaf
- `3` = mixed forest

The **primary mixed-forest definition uses hard RF labels in a spatial neighbourhood**. This keeps the ecological concept of *spatial mixture* separate from RF classification uncertainty.

The adopted manuscript setting is:

- contextual method: **hard-label composition**
- neighbourhood width: **500 m**
- dominance threshold: **0.75**

The 500 m / 0.75 setting is treated as the **nominal operational definition**, not as a statistically optimized parameter pair. Robustness is evaluated across 100–1000 m neighbourhoods and dominance thresholds 0.50–0.90.

RF class probabilities, probability-based context, entropy, and parameter-stability products are retained as **secondary uncertainty/sensitivity analyses**.

Global products are treated as **comparison products, not ground truth**. Independent observations are used for evaluation of the RF.


> **Reviewed version (v3).**  
> This revision fixes two important issues found after inspecting the first executed outputs:
>
> 1. The independent SiDroForest/PANGAEA test subset currently contains **7 evaluable Summergreen units and no Evergreen reference units**. It therefore cannot support two-class balanced accuracy, macro-F1, kappa, or Evergreen performance. This notebook now reports it explicitly as a **class-specific external evaluation** rather than forcing zero-support Evergreen metrics.
> 2. The original contextual-sensitivity implementation retained every full-resolution map in RAM. With the large SiDroForest rasters (up to ~35.6 million forest pixels), that can require many GB of memory. The revised analysis is **streamed site-by-site**: it accumulates sensitivity/stability statistics, saves only selected maps, and does not keep the full parameter cube in memory.
>
> The already generated RF probability rasters are reused automatically when present.

## v16 changes

This revision:

- makes **hard-label contextual composition** the primary mixed-forest method;
- uses **500 m / 0.75** as the nominal manuscript setting while retaining the complete sensitivity analysis;
- keeps probability-based context and entropy as secondary uncertainty analyses;
- retains the corrected LK (`LK.tif + LK2.tif`) CGLS reconstruction and scale-aware comparison;
- ensures SiDroForest is aggregated to global-product support by categorical **mode / majority**;
- generates a compact set of publication-ready main and supplementary figures;
- creates draft figure captions and a figure inventory to help reduce redundancy in the manuscript.


## Running this repository version

This copy has notebook outputs cleared and Google Drive-specific paths removed. Install `requirements.txt`, place/source the required data as described in `data/README.md`, and run from the repository root. Paths can be overridden with the environment variables listed in `.env.example`.


In [ ]:
# ============================================================
# 0. SETUP — repository / environment configuration
# ============================================================
import os
import ast
import math
import warnings
import joblib
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.ndimage import uniform_filter
from rasterio.warp import reproject, Resampling
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    cohen_kappa_score,
)

warnings.filterwarnings("ignore")

# Run the notebook from the repository root, or define these environment
# variables to point to your local copies of the source data.
REPO_ROOT = Path(os.environ.get("SIDRO_REPO_ROOT", Path.cwd())).resolve()
DATA_ROOT = Path(os.environ.get("SIDRO_DATA_ROOT", REPO_ROOT / "data")).resolve()
SIDRO = Path(os.environ.get("SIDRO_ROOT", DATA_ROOT / "Sidroforest")).resolve()
FCSROOT = Path(os.environ.get("FCS30_ROOT", DATA_ROOT / "FCS30")).resolve()
CGLS_ROOT = Path(
    os.environ.get(
        "CGLS_ROOT",
        DATA_ROOT / "globcover_6_classes_named_per_plot",
    )
).resolve()
HIST_CGLS_ROOT = Path(
    os.environ.get(
        "SIDRO_HIST_CGLS_ROOT",
        DATA_ROOT / "Cropped_Images_6_classes_downsampled",
    )
).resolve()

OUT = Path(os.environ.get("SIDRO_OUTPUT_ROOT", REPO_ROOT / "outputs")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

MODEL_FILE = Path(
    os.environ.get(
        "SIDRO_MODEL_FILE",
        DATA_ROOT / "Clustered_LS_+9_28nov" / "RF_old_full_sentinel2.pkl",
    )
).resolve()

SITE_SUMMARY = Path(
    os.environ.get(
        "SIDRO_SITE_SUMMARY",
        SIDRO / "classified_old_full_s2_RF" /
        "sidroforest_old_sentinel_RF_confidence_summary_clean.csv",
    )
).resolve()

FIELD_2021_CSV = Path(
    os.environ.get(
        "SIDRO_FIELD_2021_CSV",
        FCSROOT / "siberia_cropped_plots_RAW_RF_no_mixed_NDVI068" /
        "raw_rf_no_mixed_plot_evaluation_summary.csv",
    )
).resolve()

PANGAEA_CSV = Path(
    os.environ.get(
        "SIDRO_PANGAEA_CSV",
        FCSROOT / "pangaea_shapefiles_RAW_RF_no_mixed_NDVI068" /
        "pangaea_raw_rf_no_mixed_summergreen_evergreen_masked_excluded_summary.csv",
    )
).resolve()

print("Repository root:", REPO_ROOT)
print("Data root:", DATA_ROOT)
print("Output folder:", OUT)
print("Model exists:", MODEL_FILE.exists())
print("Site summary exists:", SITE_SUMMARY.exists())
print("2021 evaluation CSV exists:", FIELD_2021_CSV.exists())
print("PANGAEA/test CSV exists:", PANGAEA_CSV.exists())


## 1. Load the existing RF and audit the class convention

The model itself is not claimed as new. The purpose here is to retain its **class probabilities** so that the contextual method can use more information than a hard class map.

The code does not assume the order of columns returned by `predict_proba`; it finds the probability column corresponding to class 1 and class 2 from `model.classes_`.

In [ ]:
# ============================================================
# 1. LOAD MODEL
# ============================================================
bundle = joblib.load(MODEL_FILE)
RF = bundle["model"]
MODEL_FEATURE_ORDER = list(bundle["feature_order"])

print("Model:", RF)
print("Classes:", RF.classes_)
print("Feature order:", MODEL_FEATURE_ORDER)

if 1 not in RF.classes_ or 2 not in RF.classes_:
    raise ValueError(
        "Expected RF classes 1 and 2. Check the saved model before continuing."
    )

EVERGREEN = 1
SUMMERGREEN = 2
MASKED = 0
MIXED = 3

PROB_COL_EVERGREEN = list(RF.classes_).index(EVERGREEN)
PROB_COL_SUMMERGREEN = list(RF.classes_).index(SUMMERGREEN)

print("P(evergreen) column:", PROB_COL_EVERGREEN)
print("P(summergreen) column:", PROB_COL_SUMMERGREEN)

# Part A — formal independent evaluation

The existing notebooks already evaluate geographically separate data, but mainly as match tallies. Here we convert those outputs into established classification metrics.

The **polygon/plot is the evaluation unit**. We do not count every raster pixel inside a polygon as an independent observation.

For each dataset we report:

- number of evaluable plots/polygons
- overall accuracy
- balanced accuracy
- Cohen's κ
- macro F1
- precision / recall / F1 by class
- confusion matrix
- bootstrap 95% confidence interval for overall accuracy

The two datasets remain separate rather than being presented as one statistically homogeneous sample.

In [ ]:
# ============================================================
# 2. VALIDATION HELPERS — SUPPORT-AWARE
# ============================================================
LABELS = ["Evergreen", "Summergreen"]

def normalize_label(x):
    x = str(x).strip().lower()
    if "evergreen" in x:
        return "Evergreen"
    if "summergreen" in x or "larix" in x:
        return "Summergreen"
    return None

def bootstrap_accuracy_ci(y_true, y_pred, n_boot=5000, seed=42):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    scores = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[b] = np.mean(y_true[idx] == y_pred[idx])
    return np.quantile(scores, 0.025), np.quantile(scores, 0.975)

def formal_metrics(df, dataset_name):
    d = df.dropna(subset=["ground_truth", "prediction"]).copy()
    y_true = d["ground_truth"].to_numpy()
    y_pred = d["prediction"].to_numpy()

    observed_classes = [x for x in LABELS if np.sum(y_true == x) > 0]
    n_classes = len(observed_classes)

    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    p, r, f, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    ci_lo, ci_hi = bootstrap_accuracy_ci(y_true, y_pred)

    # Two-class metrics are meaningful only if BOTH reference classes occur.
    if n_classes == 2:
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        macro_f1 = np.mean(f)
        kappa = cohen_kappa_score(y_true, y_pred)
        evaluation_scope = "two-class independent evaluation"
    else:
        bal_acc = np.nan
        macro_f1 = np.nan
        kappa = np.nan
        evaluation_scope = (
            "class-specific independent evaluation: "
            + ", ".join(observed_classes)
        )

    overall = pd.DataFrame([{
        "dataset": dataset_name,
        "evaluation_scope": evaluation_scope,
        "n_units": len(d),
        "n_evergreen_reference": int(np.sum(y_true == "Evergreen")),
        "n_summergreen_reference": int(np.sum(y_true == "Summergreen")),
        "accuracy": accuracy_score(y_true, y_pred),
        "accuracy_ci95_low": ci_lo,
        "accuracy_ci95_high": ci_hi,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "cohen_kappa": kappa
    }])

    per_class = pd.DataFrame({
        "dataset": dataset_name,
        "class": LABELS,
        "precision": p,
        "recall": r,
        "f1": f,
        "support": support
    })
    # Zero support is "not evaluated", not performance = 0.
    per_class.loc[per_class["support"] == 0, ["precision","recall","f1"]] = np.nan

    cm_df = pd.DataFrame(
        cm,
        index=[f"GT_{x}" for x in LABELS],
        columns=[f"Pred_{x}" for x in LABELS]
    )
    return overall, per_class, cm_df, d

In [ ]:
# ============================================================
# 3. PREPARE 2021 FIELD EVALUATION
# ============================================================
field2021 = None

if FIELD_2021_CSV.exists():
    raw = pd.read_csv(FIELD_2021_CSV)
    print("2021 columns:", list(raw.columns))

    # The supplied notebook uses these string columns.
    # This deliberately avoids relying on older numeric class-code comments.
    if {"label_from_folder", "dominant_predicted_label"}.issubset(raw.columns):
        d = raw.copy()

        if "status" in d.columns:
            d = d[d["status"] == "success"].copy()

        d["ground_truth"] = d["label_from_folder"].apply(normalize_label)
        d["prediction"] = d["dominant_predicted_label"].apply(normalize_label)

        # Fully masked polygons are not treated as class predictions.
        d = d.dropna(subset=["ground_truth", "prediction"]).copy()

        field2021 = d
        print("Evaluable 2021 units:", len(field2021))
        display(field2021[["ground_truth", "prediction"]].head())
    else:
        print(
            "Expected columns not found. Inspect the table above and edit this cell "
            "to map its ground-truth and predicted label columns."
        )
else:
    print("2021 evaluation CSV not found.")

In [ ]:
# ============================================================
# 4. PREPARE PANGAEA / SIDROFOREST TEST EVALUATION
# ============================================================
testset = None

if PANGAEA_CSV.exists():
    raw = pd.read_csv(PANGAEA_CSV)
    print("Test-set columns:", list(raw.columns))

    if {"expected_label", "dominant_no_mask_label"}.issubset(raw.columns):
        d = raw.copy()

        if "status" in d.columns:
            d = d[d["status"] == "success"].copy()

        if "dominant_no_mask_class" in d.columns:
            d = d[d["dominant_no_mask_class"] != 0].copy()

        d["ground_truth"] = d["expected_label"].apply(normalize_label)
        d["prediction"] = d["dominant_no_mask_label"].apply(normalize_label)
        d = d.dropna(subset=["ground_truth", "prediction"]).copy()

        testset = d
        print("Evaluable test-set units:", len(testset))
        display(testset[["ground_truth", "prediction"]].head())
    else:
        print(
            "Expected columns not found. Inspect the table above and edit this cell "
            "to map its ground-truth and predicted label columns."
        )
else:
    print("PANGAEA/test CSV not found.")

In [ ]:
# ============================================================
# 4B. AUDIT WHICH INDEPENDENT UNITS WERE / WERE NOT EVALUATED
# ============================================================
print("2021 FIELD AUDIT")
if FIELD_2021_CSV.exists():
    a = pd.read_csv(FIELD_2021_CSV)
    print("Total rows:", len(a))
    if "status" in a.columns:
        print("\nStatus:")
        print(a["status"].value_counts(dropna=False))
    if "label_from_folder" in a.columns:
        print("\nReference labels:")
        print(a["label_from_folder"].value_counts(dropna=False))
    if "dominant_predicted_label" in a.columns:
        print("\nPredicted labels:")
        print(a["dominant_predicted_label"].value_counts(dropna=False))

    a["_gt_norm"] = a["label_from_folder"].apply(normalize_label)
    a["_pred_norm"] = a["dominant_predicted_label"].apply(normalize_label)
    excluded = a[
        (a.get("status", "success") != "success") |
        a["_gt_norm"].isna() |
        a["_pred_norm"].isna()
    ].copy()
    print("\nRows excluded from formal metrics:", len(excluded))
    cols = [x for x in [
        "plot","label_from_folder","dominant_predicted_label","status"
    ] if x in excluded.columns]
    display(excluded[cols])

print("\nSIDROFOREST / PANGAEA TEST AUDIT")
if PANGAEA_CSV.exists():
    a = pd.read_csv(PANGAEA_CSV)
    print("Total rows:", len(a))
    if "status" in a.columns:
        print("\nStatus:")
        print(a["status"].value_counts(dropna=False))
    if "expected_label" in a.columns:
        print("\nReference labels:")
        print(a["expected_label"].value_counts(dropna=False))
    if "dominant_no_mask_label" in a.columns:
        print("\nPredicted labels:")
        print(a["dominant_no_mask_label"].value_counts(dropna=False))
    if "dominant_no_mask_class" in a.columns:
        print("\nDominant predicted class:")
        print(a["dominant_no_mask_class"].value_counts(dropna=False))

print(
    "\nUse this audit in the manuscript workflow: explain why any independent "
    "units are non-evaluable rather than silently dropping them."
)

In [ ]:
# ============================================================
# 5. CALCULATE + SAVE SUPPORT-AWARE VALIDATION METRICS
# ============================================================
validation_overall = []
validation_per_class = []
validation_confusions = {}

for dataset_name, d in [
    ("Independent_2021_field", field2021),
    ("Independent_SiDroForest_test", testset)
]:
    if d is None or len(d) == 0:
        continue

    overall, per_class, cm, detail = formal_metrics(d, dataset_name)
    validation_overall.append(overall)
    validation_per_class.append(per_class)
    validation_confusions[dataset_name] = cm

    print("\n" + "=" * 70)
    print(dataset_name)
    display(overall.round(3))
    display(per_class.round(3))
    display(cm)

    observed = [x for x in LABELS if (detail["ground_truth"] == x).any()]
    print("Reference classes actually represented:", observed)
    print(classification_report(
        detail["ground_truth"], detail["prediction"],
        labels=observed, zero_division=0
    ))

    cm.to_csv(OUT / f"{dataset_name}_confusion_matrix.csv")
    detail.to_csv(OUT / f"{dataset_name}_evaluation_units.csv", index=False)

if validation_overall:
    validation_overall_df = pd.concat(validation_overall, ignore_index=True)
    validation_per_class_df = pd.concat(validation_per_class, ignore_index=True)

    validation_overall_df.to_csv(
        OUT / "independent_validation_overall_metrics.csv", index=False
    )
    validation_per_class_df.to_csv(
        OUT / "independent_validation_per_class_metrics.csv", index=False
    )

    display(validation_overall_df.round(3))
    display(validation_per_class_df.round(3))

    print("\nINTERPRETATION NOTE:")
    print(
        "The 2021 set currently supports a two-class evaluation (2 Evergreen, 7 Summergreen). "
        "The SiDroForest/PANGAEA subset currently supports Summergreen-only external evaluation "
        "(7 Summergreen, 0 Evergreen). Do not describe the latter as a two-class validation."
    )

In [ ]:
# ============================================================
# 6. VALIDATION FIGURE
# ============================================================
if validation_per_class:
    fig_df = validation_per_class_df.copy()

    x = np.arange(len(LABELS))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))

    datasets = fig_df["dataset"].unique()

    for i, ds in enumerate(datasets):
        sub = fig_df[fig_df["dataset"] == ds].set_index("class").reindex(LABELS)
        ax.bar(
            x + (i - (len(datasets)-1)/2) * width,
            sub["f1"],
            width,
            label=ds
        )

    ax.set_xticks(x)
    ax.set_xticklabels(LABELS)
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1 score")
    ax.set_title("Independent evaluation by forest class")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT / "independent_validation_class_f1.png", dpi=300)
    plt.show()

# Part B — regenerate class-probability maps for the existing SiDroForest sites

The existing site summary contains the raster stack used for each site. This section uses the **saved RF model's own feature order** and raster band descriptions to create:

- `P(evergreen)`
- `P(summergreen)`
- hard RF class
- maximum RF probability

This allows us to compare the old hard-label contextual method with a new **probability-based contextual method**.

No new classifier is trained here.

In [ ]:
# ============================================================
# 7. FEATURE EXTRACTION FROM EXISTING STACKS
# ============================================================
def canonical_band_name(x):
    x = str(x).strip().upper()
    aliases = {
        "B2": "B02", "B02": "B02",
        "B3": "B03", "B03": "B03",
        "B4": "B04", "B04": "B04",
        "B5": "B05", "B05": "B05",
        "B6": "B06", "B06": "B06",
        "B7": "B07", "B07": "B07",
        "B8": "B08", "B08": "B08",
        "B8A": "B8A",
        "B11": "B11",
        "B12": "B12",
        "NDVI": "NDVI"
    }
    return aliases.get(x, x)

MODEL_FEATURES_CANONICAL = [canonical_band_name(x) for x in MODEL_FEATURE_ORDER]
print("Canonical model order:", MODEL_FEATURES_CANONICAL)

def read_model_features_from_stack(stack_file):
    with rasterio.open(stack_file) as src:
        data = src.read().astype("float32")
        profile = src.profile.copy()
        descriptions = [
            canonical_band_name(d) if d is not None else None
            for d in src.descriptions
        ]

        if src.nodata is not None:
            data[np.isclose(data, src.nodata)] = np.nan

    # Prefer saved raster band descriptions.
    if all(f in descriptions for f in MODEL_FEATURES_CANONICAL):
        idx = [descriptions.index(f) for f in MODEL_FEATURES_CANONICAL]
        model_stack = data[idx]
    else:
        raise ValueError(
            f"Cannot reconstruct model feature order for {stack_file}. "
            f"Raster descriptions={descriptions}; model expects={MODEL_FEATURES_CANONICAL}"
        )

    return model_stack, profile

def read_mask(mask_file, ref_profile):
    with rasterio.open(mask_file) as src:
        if (
            src.crs == ref_profile["crs"] and
            src.transform == ref_profile["transform"] and
            src.width == ref_profile["width"] and
            src.height == ref_profile["height"]
        ):
            return src.read(1) > 0

        dst = np.zeros(
            (ref_profile["height"], ref_profile["width"]),
            dtype="uint8"
        )
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            resampling=Resampling.nearest
        )
        return dst > 0

In [ ]:
# ============================================================
# 8. GENERATE RF PROBABILITY MAPS
# ============================================================
PROB_DIR = OUT / "rf_probability_maps"
PROB_DIR.mkdir(parents=True, exist_ok=True)

def write_float_map(path, arr, profile):
    p = profile.copy()
    for k in ["blockxsize", "blockysize", "tiled", "interleave", "photometric"]:
        p.pop(k, None)
    p.update(count=1, dtype="float32", nodata=np.nan, compress="lzw")
    with rasterio.open(path, "w", **p) as dst:
        dst.write(arr.astype("float32"), 1)

def write_uint8_map(path, arr, profile):
    p = profile.copy()
    for k in ["blockxsize", "blockysize", "tiled", "interleave", "photometric"]:
        p.pop(k, None)
    p.update(count=1, dtype="uint8", nodata=0, compress="lzw")
    with rasterio.open(path, "w", **p) as dst:
        dst.write(arr.astype("uint8"), 1)

def probability_maps_for_site(site, stack_file, mask_file, chunk_size=500_000):
    model_stack, profile = read_model_features_from_stack(stack_file)
    forest_mask = read_mask(mask_file, profile)

    h, w = forest_mask.shape
    X = np.moveaxis(model_stack, 0, -1).reshape(-1, model_stack.shape[0])

    valid = np.all(np.isfinite(X), axis=1) & forest_mask.reshape(-1)
    valid_idx = np.where(valid)[0]

    p_eg = np.full(X.shape[0], np.nan, dtype="float32")
    p_sg = np.full(X.shape[0], np.nan, dtype="float32")
    hard = np.zeros(X.shape[0], dtype="uint8")

    for start in range(0, len(valid_idx), chunk_size):
        idx = valid_idx[start:start+chunk_size]
        probs = RF.predict_proba(X[idx])

        p_eg[idx] = probs[:, PROB_COL_EVERGREEN]
        p_sg[idx] = probs[:, PROB_COL_SUMMERGREEN]
        hard[idx] = RF.classes_[np.argmax(probs, axis=1)].astype("uint8")

    p_eg = p_eg.reshape(h, w)
    p_sg = p_sg.reshape(h, w)
    hard = hard.reshape(h, w)
    max_prob = np.fmax(p_eg, p_sg)

    eg_file = PROB_DIR / f"{site}_P_evergreen.tif"
    sg_file = PROB_DIR / f"{site}_P_summergreen.tif"
    hard_file = PROB_DIR / f"{site}_hard_RF.tif"
    conf_file = PROB_DIR / f"{site}_max_probability.tif"

    write_float_map(eg_file, p_eg, profile)
    write_float_map(sg_file, p_sg, profile)
    write_float_map(conf_file, max_prob, profile)
    write_uint8_map(hard_file, hard, profile)

    return {
        "site": site,
        "P_evergreen_file": str(eg_file),
        "P_summergreen_file": str(sg_file),
        "hard_file": str(hard_file),
        "max_probability_file": str(conf_file),
        "forest_pixels": int(forest_mask.sum()),
        "mean_max_probability": float(np.nanmean(max_prob))
    }

prob_rows = []

if SITE_SUMMARY.exists():
    site_summary = pd.read_csv(SITE_SUMMARY)

    success = site_summary[
        site_summary["status"].astype(str).str.lower().eq("success")
    ].copy()

    for _, row in success.iterrows():
        site = row["site"]
        stack_file = Path(row["stack_file"])
        mask_file = Path(row["mask_file"])

        print("\nProbability maps:", site)

        if not stack_file.exists() or not mask_file.exists():
            print("Missing stack or mask; skipping.")
            continue

        # Reuse the expensive outputs from a previous run when all four exist.
        eg_file = PROB_DIR / f"{site}_P_evergreen.tif"
        sg_file = PROB_DIR / f"{site}_P_summergreen.tif"
        hard_file = PROB_DIR / f"{site}_hard_RF.tif"
        conf_file = PROB_DIR / f"{site}_max_probability.tif"

        if all(p.exists() for p in [eg_file, sg_file, hard_file, conf_file]):
            with rasterio.open(eg_file) as src:
                pe = src.read(1)
            with rasterio.open(conf_file) as src:
                mp = src.read(1)
            prob_rows.append({
                "site": site,
                "P_evergreen_file": str(eg_file),
                "P_summergreen_file": str(sg_file),
                "hard_file": str(hard_file),
                "max_probability_file": str(conf_file),
                "forest_pixels": int(np.isfinite(pe).sum()),
                "mean_max_probability": float(np.nanmean(mp))
            })
            print("Reused existing probability rasters.")
            continue

        try:
            prob_rows.append(
                probability_maps_for_site(site, stack_file, mask_file)
            )
        except Exception as e:
            print("FAILED:", site, e)

probability_summary = pd.DataFrame(prob_rows)
probability_summary.to_csv(OUT / "rf_probability_map_summary.csv", index=False)
display(probability_summary)

# Part C — contextual forest composition

Two formulations are retained, but they have different roles.

### A. Hard-label spatial composition — **primary method**

For each neighbourhood, calculate the local proportions of confidently classified evergreen and summergreen RF pixels. Mixed forest therefore represents the **co-occurrence of the two forest types in space**.

### B. Probability-based context — **secondary sensitivity analysis**

The local mean RF probabilities are also calculated:

\[
\bar{P}_E(N)=\frac{1}{|N|}\sum_{i \in N}P_E(i)
\]

\[
\bar{P}_S(N)=\frac{1}{|N|}\sum_{i \in N}P_S(i)
\]

This is useful for uncertainty analysis, but it is not used as the primary ecological definition of mixed forest because probability averaging can combine **classifier uncertainty** with **spatial mixture**.


In [ ]:
# ============================================================
# 9. CONTEXTUAL FUNCTIONS
# ============================================================
WINDOW_WIDTHS_M = [100, 250, 500, 750, 1000]
DOMINANCE_THRESHOLDS = [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]

def odd_window_pixels(width_m, pixel_size_m):
    n = max(1, int(round(width_m / pixel_size_m)))
    if n % 2 == 0:
        n += 1
    return n

def local_mean_valid(values, valid, window_pixels):
    values0 = np.where(valid, values, 0).astype("float32")
    valid0 = valid.astype("float32")

    num = uniform_filter(
        values0, size=window_pixels, mode="constant", cval=0
    )
    den = uniform_filter(
        valid0, size=window_pixels, mode="constant", cval=0
    )

    out = np.divide(
        num, den,
        out=np.full_like(num, np.nan),
        where=den > 0
    )
    return out

def hard_context(hard, valid, window_pixels):
    eg = ((hard == EVERGREEN) & valid).astype("float32")
    sg = ((hard == SUMMERGREEN) & valid).astype("float32")

    eg_share = local_mean_valid(eg, valid, window_pixels)
    sg_share = local_mean_valid(sg, valid, window_pixels)

    total = eg_share + sg_share
    eg_share = np.divide(eg_share, total, out=np.full_like(eg_share, np.nan), where=total > 0)
    sg_share = np.divide(sg_share, total, out=np.full_like(sg_share, np.nan), where=total > 0)

    return eg_share, sg_share

def probability_context(p_eg, p_sg, valid, window_pixels):
    eg_mean = local_mean_valid(p_eg, valid & np.isfinite(p_eg), window_pixels)
    sg_mean = local_mean_valid(p_sg, valid & np.isfinite(p_sg), window_pixels)

    total = eg_mean + sg_mean
    eg_share = np.divide(eg_mean, total, out=np.full_like(eg_mean, np.nan), where=total > 0)
    sg_share = np.divide(sg_mean, total, out=np.full_like(sg_mean, np.nan), where=total > 0)

    return eg_share, sg_share

def categorical_from_composition(eg_share, sg_share, valid, threshold):
    out = np.zeros(valid.shape, dtype="uint8")

    out[valid & (eg_share >= threshold)] = EVERGREEN
    out[valid & (sg_share >= threshold)] = SUMMERGREEN

    mixed = (
        valid &
        np.isfinite(eg_share) &
        np.isfinite(sg_share) &
        (eg_share < threshold) &
        (sg_share < threshold)
    )
    out[mixed] = MIXED

    return out

def normalized_entropy(eg_share, sg_share):
    eps = 1e-12
    pe = np.clip(eg_share, eps, 1)
    ps = np.clip(sg_share, eps, 1)

    H = -(pe*np.log(pe) + ps*np.log(ps)) / np.log(2)
    H[~np.isfinite(eg_share) | ~np.isfinite(sg_share)] = np.nan
    return H

In [ ]:
# ============================================================
# 10. FINAL CONTEXTUAL ANALYSIS
# ============================================================
CONTEXT_DIR = OUT / "contextual_maps"
CONTEXT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_SITES = {"LK", "YA"}

# PRIMARY MANUSCRIPT SETTING
ADOPTED_METHOD = "hard_label"
ADOPTED_SCALE_M = 500
ADOPTED_THRESHOLD = 0.75

SENSITIVITY_CSV = OUT / "contextual_method_full_sensitivity.csv"
STABILITY_CSV = OUT / "parameter_stability_summary.csv"

context_rows = []
stability_rows = []
adopted_map_files = {}

# Reuse existing full sensitivity summaries where possible.
if SENSITIVITY_CSV.exists() and STABILITY_CSV.exists():
    print("Reusing existing contextual sensitivity summaries.")
    context_df = pd.read_csv(SENSITIVITY_CSV)
    stability_df = pd.read_csv(STABILITY_CSV)

else:
    print("Sensitivity summaries not found; calculating full sensitivity grid.")

    for _, row in probability_summary.iterrows():
        site = row["site"]
        print("\nProcessing contextual sensitivity:", site)

        with rasterio.open(row["P_evergreen_file"]) as src:
            p_eg = src.read(1).astype("float32")
            profile = src.profile.copy()
            pixel_size_m = abs(src.transform.a)

        with rasterio.open(row["P_summergreen_file"]) as src:
            p_sg = src.read(1).astype("float32")

        with rasterio.open(row["hard_file"]) as src:
            hard = src.read(1).astype("uint8")

        valid = (
            np.isfinite(p_eg)
            & np.isfinite(p_sg)
            & np.isin(hard, [EVERGREEN, SUMMERGREEN])
        )
        forest_n = int(valid.sum())

        mixed_hits = {
            "hard_label": np.zeros(valid.shape, dtype="uint8"),
            "probability": np.zeros(valid.shape, dtype="uint8"),
        }
        n_combinations = 0

        for width_m in WINDOW_WIDTHS_M:
            wpx = odd_window_pixels(width_m, pixel_size_m)
            print(f"  {width_m} m -> {wpx} pixels")

            hard_eg, hard_sg = hard_context(hard, valid, wpx)
            prob_eg, prob_sg = probability_context(
                p_eg, p_sg, valid, wpx
            )

            entropy = normalized_entropy(prob_eg, prob_sg)
            entropy[~valid] = np.nan

            if site in SAVE_SITES:
                entropy_file = (
                    CONTEXT_DIR / f"{site}_entropy_{width_m}m.tif"
                )
                write_float_map(entropy_file, entropy, profile)

            for method, eg_share, sg_share in [
                ("hard_label", hard_eg, hard_sg),
                ("probability", prob_eg, prob_sg),
            ]:
                for threshold in DOMINANCE_THRESHOLDS:
                    cat = categorical_from_composition(
                        eg_share, sg_share, valid, threshold
                    )

                    mixed_hits[method] += (
                        cat == MIXED
                    ).astype("uint8")

                    n_eg = int(np.sum((cat == EVERGREEN) & valid))
                    n_sg = int(np.sum((cat == SUMMERGREEN) & valid))
                    n_mx = int(np.sum((cat == MIXED) & valid))

                    context_rows.append({
                        "site": site,
                        "method": method,
                        "window_width_m": width_m,
                        "window_pixels": wpx,
                        "threshold": threshold,
                        "forest_pixels": forest_n,
                        "evergreen_pct": (
                            n_eg / forest_n * 100 if forest_n else np.nan
                        ),
                        "summergreen_pct": (
                            n_sg / forest_n * 100 if forest_n else np.nan
                        ),
                        "mixed_pct": (
                            n_mx / forest_n * 100 if forest_n else np.nan
                        ),
                        "mean_entropy": float(np.nanmean(entropy)),
                    })

            n_combinations += len(DOMINANCE_THRESHOLDS)
            del hard_eg, hard_sg, prob_eg, prob_sg, entropy

        for method in ["hard_label", "probability"]:
            stability = (
                mixed_hits[method].astype("float32") / n_combinations
            )
            stability[~valid] = np.nan
            vals = stability[np.isfinite(stability)]

            stability_rows.append({
                "site": site,
                "method": method,
                "n_parameter_combinations": n_combinations,
                "mean_mixed_stability": float(np.mean(vals)),
                "stable_mixed_ge_0p8_pct":
                    float(np.mean(vals >= 0.8) * 100),
                "stable_nonmixed_le_0p2_pct":
                    float(np.mean(vals <= 0.2) * 100),
                "parameter_sensitive_0p2_to_0p8_pct":
                    float(np.mean((vals > 0.2) & (vals < 0.8)) * 100),
            })

            if site in SAVE_SITES:
                st_file = (
                    CONTEXT_DIR / f"{site}_{method}_mixed_stability.tif"
                )
                write_float_map(st_file, stability, profile)

        del p_eg, p_sg, hard, valid, mixed_hits

    context_df = pd.DataFrame(context_rows)
    stability_df = pd.DataFrame(stability_rows)

    context_df.to_csv(SENSITIVITY_CSV, index=False)
    stability_df.to_csv(STABILITY_CSV, index=False)

# Always generate/refresh the adopted HARD-LABEL maps for LK/YA.
print("\nGenerating adopted hard-label contextual maps:")
for site in sorted(SAVE_SITES):
    hit = probability_summary[
        probability_summary["site"] == site
    ]

    if hit.empty:
        print(site, ": no probability-summary entry; skipped.")
        continue

    row = hit.iloc[0]

    with rasterio.open(row["hard_file"]) as src:
        hard = src.read(1).astype("uint8")
        profile = src.profile.copy()
        pixel_size_m = abs(src.transform.a)

    valid = np.isin(hard, [EVERGREEN, SUMMERGREEN])
    wpx = odd_window_pixels(ADOPTED_SCALE_M, pixel_size_m)

    eg_share, sg_share = hard_context(hard, valid, wpx)

    cat = categorical_from_composition(
        eg_share,
        sg_share,
        valid,
        ADOPTED_THRESHOLD
    )

    out_file = CONTEXT_DIR / (
        f"{site}_{ADOPTED_METHOD}_{ADOPTED_SCALE_M}m_"
        f"t{ADOPTED_THRESHOLD:.2f}.tif"
    )
    write_uint8_map(out_file, cat, profile)
    adopted_map_files[site] = str(out_file)

    forest = np.isin(cat, [1, 2, 3])
    print(
        site,
        "| saved:", out_file.name,
        "| evergreen:",
        round(float(np.mean(cat[forest] == EVERGREEN) * 100), 2),
        "% | summergreen:",
        round(float(np.mean(cat[forest] == SUMMERGREEN) * 100), 2),
        "% | mixed:",
        round(float(np.mean(cat[forest] == MIXED) * 100), 2),
        "%"
    )

display(
    context_df[
        (context_df["site"].isin(["LK", "YA"]))
        & (context_df["method"] == "hard_label")
        & (context_df["window_width_m"] == ADOPTED_SCALE_M)
        & np.isclose(context_df["threshold"], ADOPTED_THRESHOLD)
    ].round(3)
)

display(
    stability_df[
        stability_df["site"].isin(["LK", "YA"])
    ].round(3)
)

print("Primary manuscript maps:", adopted_map_files)


## 10A. Window × threshold sensitivity — main manuscript figure

The primary sensitivity figure uses the **hard-label contextual method only**, because this is the adopted ecological definition of spatial forest composition.

The probability-based contextual formulation is retained in the supplementary material as an uncertainty/sensitivity comparison.


In [ ]:
# ============================================================
# 11. MAIN FIGURE — HARD-LABEL WINDOW × THRESHOLD SENSITIVITY
# ============================================================
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

MANUSCRIPT_FIG_DIR = OUT / "manuscript_figures_v16"
MANUSCRIPT_FIG_DIR.mkdir(parents=True, exist_ok=True)

sites = ["LK", "YA"]

fig, axes = plt.subplots(
    1, 2, figsize=(12.0, 4.8),
    constrained_layout=True
)

for ax, site in zip(axes, sites):
    sub = context_df[
        (context_df["site"] == site)
        & (context_df["method"] == "hard_label")
    ].copy()

    pivot = sub.pivot(
        index="window_width_m",
        columns="threshold",
        values="mixed_pct"
    ).sort_index()

    im = ax.imshow(
        pivot.values,
        aspect="auto",
        vmin=0,
        vmax=100
    )

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{x:.2f}" for x in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{int(x)}" for x in pivot.index])

    ax.set_xlabel("Dominance threshold")
    ax.set_ylabel("Neighbourhood width (m)")
    ax.set_title("Lake Khamra" if site == "LK" else "Yakutsk")

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(
                j, i, f"{pivot.iloc[i, j]:.1f}",
                ha="center", va="center", fontsize=7
            )

    y_idx = list(pivot.index).index(ADOPTED_SCALE_M)
    x_idx = list(pivot.columns).index(ADOPTED_THRESHOLD)
    ax.add_patch(
        plt.Rectangle(
            (x_idx - 0.48, y_idx - 0.48),
            0.96, 0.96,
            fill=False,
            linewidth=2.5
        )
    )

cbar = fig.colorbar(im, ax=axes, shrink=0.88, pad=0.02)
cbar.set_label("Mixed forest (%)")

fig.suptitle(
    "Sensitivity of hard-label mixed-forest extent to "
    "neighbourhood width and dominance threshold",
    fontsize=12
)

png = MANUSCRIPT_FIG_DIR / "Fig_sensitivity_hard_label.png"
pdf = MANUSCRIPT_FIG_DIR / "Fig_sensitivity_hard_label.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


# Part D — parameter stability (supplementary)

For each forest pixel, parameter stability is the fraction of tested neighbourhood × threshold combinations in which the pixel is classified as mixed.

The primary supplementary maps show the **hard-label method**. Probability-based stability is summarized separately to show why probability averaging is retained as an uncertainty analysis rather than the primary ecological definition.


In [ ]:
# ============================================================
# 12. SUPPLEMENTARY FIGURE — HARD-LABEL PARAMETER STABILITY
# ============================================================
fig, axes = plt.subplots(
    1, 2, figsize=(11, 5.2),
    constrained_layout=True
)

last_im = None

for ax, site in zip(axes, ["LK", "YA"]):
    st_file = CONTEXT_DIR / f"{site}_hard_label_mixed_stability.tif"

    if not st_file.exists():
        ax.text(
            0.5, 0.5,
            f"{site}: stability raster not found",
            ha="center", va="center",
            transform=ax.transAxes
        )
        ax.axis("off")
        continue

    with rasterio.open(st_file) as src:
        stability = src.read(1).astype("float32")

    stability[~np.isfinite(stability)] = np.nan

    last_im = ax.imshow(stability, vmin=0, vmax=1)
    ax.set_title("Lake Khamra" if site == "LK" else "Yakutsk")
    ax.axis("off")

if last_im is not None:
    cbar = fig.colorbar(last_im, ax=axes, shrink=0.85, pad=0.02)
    cbar.set_label(
        "Fraction of parameter combinations classified as mixed"
    )

fig.suptitle(
    "Hard-label mixed-forest parameter stability",
    fontsize=12
)

png = MANUSCRIPT_FIG_DIR / "FigS_hard_label_parameter_stability.png"
pdf = MANUSCRIPT_FIG_DIR / "FigS_hard_label_parameter_stability.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)

display(
    stability_df[
        stability_df["site"].isin(["LK", "YA"])
    ][[
        "site",
        "method",
        "stable_mixed_ge_0p8_pct",
        "stable_nonmixed_le_0p2_pct",
        "parameter_sensitive_0p2_to_0p8_pct",
    ]].round(2)
)


# Part E — continuous probability-based mixedness (supplementary)

Normalized Shannon entropy is retained as a continuous **uncertainty / compositional ambiguity** product.

Because probability-based composition can combine classification uncertainty with true spatial mixture, this is **not** the primary mixed-forest definition in the revised manuscript.


In [ ]:
# ============================================================
# 13. SUPPLEMENTARY FIGURE — CONTINUOUS PROBABILITY ENTROPY
# ============================================================
fig, axes = plt.subplots(
    1, 2, figsize=(11, 5.2),
    constrained_layout=True
)

last_im = None

for ax, site in zip(axes, ["LK", "YA"]):
    entropy_file = CONTEXT_DIR / f"{site}_entropy_{ADOPTED_SCALE_M}m.tif"

    if not entropy_file.exists():
        ax.text(
            0.5, 0.5,
            f"{site}: entropy raster not found",
            ha="center", va="center",
            transform=ax.transAxes
        )
        ax.axis("off")
        continue

    with rasterio.open(entropy_file) as src:
        H = src.read(1).astype("float32")

    H[~np.isfinite(H)] = np.nan

    last_im = ax.imshow(H, vmin=0, vmax=1)
    ax.set_title("Lake Khamra" if site == "LK" else "Yakutsk")
    ax.axis("off")

if last_im is not None:
    cbar = fig.colorbar(last_im, ax=axes, shrink=0.85, pad=0.02)
    cbar.set_label("Normalized probability-composition entropy")

fig.suptitle(
    f"Probability-based compositional entropy "
    f"({ADOPTED_SCALE_M} m neighbourhood)",
    fontsize=12
)

png = MANUSCRIPT_FIG_DIR / "FigS_probability_entropy.png"
pdf = MANUSCRIPT_FIG_DIR / "FigS_probability_entropy.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


# Part F — does RF uncertainty relate to independent errors?

If the independent evaluation CSVs contain cropped raster files but not RF probability values, a full probability-at-polygon analysis requires cropping the new probability maps to the same polygons.

As an immediately reproducible first step, this section prepares the validation-unit tables and leaves a clear hook for attaching mean RF probability per polygon.

**Do not interpret max probability as calibrated probability without a calibration test.** Here it is used as a model-confidence score.

In [ ]:
# ============================================================
# 14. OPTIONAL CONFIDENCE / ERROR ANALYSIS TEMPLATE
# ============================================================
print(
    "Probability maps have been generated for the main SiDroForest sites. "
    "To test whether errors occur at lower RF confidence, the next step is to "
    "spatially intersect these probability rasters with the independent evaluation polygons."
)

# Once a table has columns:
#   ground_truth, prediction, mean_max_probability
# the following analysis can be used:
#
# eval_conf["correct"] = eval_conf["ground_truth"] == eval_conf["prediction"]
# display(
#     eval_conf.groupby("correct")["mean_max_probability"]
#     .agg(["count", "mean", "median", "std"])
# )
#
# A Mann-Whitney test can then compare confidence for correct vs incorrect units,
# provided there are enough errors to make that comparison meaningful.

# Part F2 — abstention-aware independent validation

The 2021 field evaluation contains reference forest observations for which the classifier can return **Masked/uncertain** rather than evergreen or summergreen. These cases should not simply disappear from the validation denominator.

We therefore report two complementary quantities:

1. **Coverage / abstention:** the proportion of independent reference units for which the classifier makes a forest-type prediction (`evergreen` or `summergreen`) versus returning `masked/uncertain`.
2. **Conditional classification performance:** confusion-matrix metrics calculated only among units receiving a forest-type prediction.

This prevents abstentions from being counted as successful classifications while also avoiding the misleading interpretation that `masked` is a third ecological forest class.

In [ ]:
# ============================================================
# ABSTENTION-AWARE VALIDATION SUMMARY — v10
# ============================================================
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, f1_score
)

def _norm_forest_label(v):
    if pd.isna(v):
        return "unknown"
    x = str(v).strip().lower()
    if x in {"evergreen", "evergreen needleleaf", "evergreen forest",
             "2", "2.0"}:
        return "evergreen"
    if x in {"summergreen", "summergreen needleleaf",
             "deciduous", "deciduous needleleaf",
             "1", "1.0"}:
        return "summergreen"
    if x in {"masked", "0", "0.0", "no forest prediction / all masked",
             "masked / no forest prediction"}:
        return "masked"
    return x

def summarize_abstention_validation(df, ref_col, pred_col, dataset_name,
                                    status_col=None, success_value="success"):
    d = df.copy()

    # If a status column exists, only geometries that were actually evaluated
    # against a raster belong in the classifier coverage denominator.
    if status_col and status_col in d.columns:
        d = d[d[status_col].astype(str).str.lower() == success_value.lower()].copy()

    d["_reference"] = d[ref_col].map(_norm_forest_label)
    d["_prediction"] = d[pred_col].map(_norm_forest_label)

    d = d[d["_reference"].isin(["evergreen", "summergreen"])].copy()
    predicted = d["_prediction"].isin(["evergreen", "summergreen"])
    abstained = d["_prediction"].eq("masked")

    n = len(d)
    n_pred = int(predicted.sum())
    n_abs = int(abstained.sum())

    row = {
        "dataset": dataset_name,
        "n_successful_reference_forest_units": n,
        "n_forest_type_predictions": n_pred,
        "n_masked_abstentions": n_abs,
        "coverage": n_pred/n if n else np.nan,
        "abstention_rate": n_abs/n if n else np.nan
    }

    cond = d.loc[predicted].copy()
    if len(cond):
        y_true = cond["_reference"]
        y_pred = cond["_prediction"]
        labels = ["evergreen", "summergreen"]

        row["conditional_accuracy"] = accuracy_score(y_true, y_pred)

        # Balanced accuracy / macro-F1 are meaningful only if both reference
        # classes are represented in this conditional subset.
        if y_true.nunique() == 2:
            row["conditional_balanced_accuracy"] = balanced_accuracy_score(
                y_true, y_pred
            )
            row["conditional_macro_f1"] = f1_score(
                y_true, y_pred, labels=labels,
                average="macro", zero_division=0
            )
        else:
            row["conditional_balanced_accuracy"] = np.nan
            row["conditional_macro_f1"] = np.nan

        print(f"\n{dataset_name}")
        print("Conditional confusion matrix:")
        display(pd.DataFrame(
            confusion_matrix(y_true, y_pred, labels=labels),
            index=[f"ref_{x}" for x in labels],
            columns=[f"pred_{x}" for x in labels]
        ))

    print("\nCoverage / abstention:")
    display(pd.DataFrame([row]).round(3))

    if n_abs:
        print("\nMasked/abstained units:")
        display(d.loc[abstained])

    return row, d

abstention_summaries = []

# 2021 polygons: use the columns already confirmed by the executed notebook.
if FIELD_2021_CSV.exists():
    field2021_raw = pd.read_csv(FIELD_2021_CSV)
    s, field2021_audit = summarize_abstention_validation(
        field2021_raw,
        ref_col="expected_class",
        pred_col="dominant_predicted_class",
        dataset_name="2021 independent field polygons"
    )
    abstention_summaries.append(s)

# SiDroForest/PANGAEA independent test:
# The supplied notebook audit shows 103 rows:
# 48 excluded non-target classes, 43 outside_raster, 12 success.
# Only status=='success' enters the coverage denominator.
# Prediction is dominant_no_mask_label/class, NOT pangaea_label.
if PANGAEA_CSV.exists():
    pangaea_raw = pd.read_csv(PANGAEA_CSV)

    # Prefer human-readable labels where available.
    ref_col = "expected_label" if "expected_label" in pangaea_raw.columns else "pangaea_label"
    pred_col = (
        "dominant_no_mask_label"
        if "dominant_no_mask_label" in pangaea_raw.columns
        else "dominant_no_mask_class"
    )

    print("\nPANGAEA status audit (outside_raster is NOT an abstention):")
    if "status" in pangaea_raw.columns:
        display(pangaea_raw["status"].value_counts(dropna=False).rename("n").to_frame())

    s, pangaea_audit = summarize_abstention_validation(
        pangaea_raw,
        ref_col=ref_col,
        pred_col=pred_col,
        dataset_name="Independent SiDroForest/PANGAEA test locations",
        status_col="status"
    )
    abstention_summaries.append(s)

abstention_validation_df = pd.DataFrame(abstention_summaries)
display(abstention_validation_df.round(3))
abstention_validation_df.to_csv(
    OUT / "independent_validation_abstention_aware_v10.csv",
    index=False
)

# Part G — final scale-aware global-product comparison (v16)

The **hard-label 500 m / 0.75 SiDroForest contextual maps** are the primary maps used here.

- FCS30 is compared on bounded native-source product grids.
- YA CGLS uses the verified non-downsampled projected YA source grid.
- LK CGLS uses the historical `LK.tif + LK2.tif` merged derivative, aggregated by categorical majority back to **nominal 100 m support**.

In every quantitative comparison, the finer SiDroForest map is reduced to the comparison grid by categorical **mode / majority class**. Global products are never treated as ground truth, and agreement metrics are not described as validation accuracy.


In [ ]:
# ============================================================
# 15. PATHS AND HELPERS — v9
# ============================================================
from pathlib import Path
from rasterio.warp import transform_bounds
from rasterio.merge import merge

FCS30_ROOT = FCSROOT
# CGLS_ROOT is configured in the setup cell above.
COMPARE_SITES = ["LK", "YA"]

def site_context_file(site):
    return CONTEXT_DIR / (
        f"{site}_{ADOPTED_METHOD}_{ADOPTED_SCALE_M}m_"
        f"t{ADOPTED_THRESHOLD:.2f}.tif"
    )

def all_tifs(root):
    return sorted(list(root.rglob("*.tif")) + list(root.rglob("*.tiff")))

# IMPORTANT:
# Native FCS30 source tiles have the original GLC_FCS30_2020_* naming.
# Do not treat other rasters stored under /FCS30 as native product tiles.
FCS30_ALL = all_tifs(FCS30_ROOT)
FCS30_NATIVE_CANDIDATES = sorted(
    p for p in FCS30_ROOT.rglob("GLC_FCS30_2020_*.tif")
    if p.is_file()
)

CGLS_ALL = all_tifs(CGLS_ROOT)

print("FCS30 total TIFFs under root:", len(FCS30_ALL))
print("Native FCS30 source tiles matching GLC_FCS30_2020_*.tif:",
      len(FCS30_NATIVE_CANDIDATES))
print("CGLS candidate TIFFs:", len(CGLS_ALL))

if not FCS30_NATIVE_CANDIDATES:
    raise FileNotFoundError(
        "No native GLC_FCS30_2020_*.tif source tiles were found."
    )

print("\nExample native FCS30 source tiles:")
for p in FCS30_NATIVE_CANDIDATES[:20]:
    print(" ", p.relative_to(FCS30_ROOT))


In [ ]:
# ============================================================
# 16. GENERIC SPATIAL AUDIT
# ============================================================
def metadata(path):
    with rasterio.open(path) as src:
        return {
            "path": str(path),
            "crs": src.crs,
            "bounds": src.bounds,
            "transform": src.transform,
            "xres": abs(src.transform.a),
            "yres": abs(src.transform.e),
            "width": src.width,
            "height": src.height,
            "nodata": src.nodata
        }

def overlap_fraction(candidate_path, reference_path):
    with rasterio.open(reference_path) as ref, rasterio.open(candidate_path) as cand:
        if ref.crs is None or cand.crs is None:
            return np.nan
        cb = transform_bounds(cand.crs, ref.crs, *cand.bounds, densify_pts=21)
        rb = ref.bounds
        left, bottom = max(cb[0],rb.left), max(cb[1],rb.bottom)
        right, top = min(cb[2],rb.right), min(cb[3],rb.top)
        if right <= left or top <= bottom:
            return 0.0
        inter = (right-left)*(top-bottom)
        ref_area = (rb.right-rb.left)*(rb.top-rb.bottom)
        return inter/ref_area

def pixel_anisotropy(xres, yres):
    if min(xres,yres) == 0:
        return np.inf
    return max(xres,yres)/min(xres,yres)

def unique_values_sample(path):
    with rasterio.open(path) as src:
        # Read at reduced resolution for diagnostics if raster is large.
        scale = max(1, int(max(src.width,src.height)/2000))
        out_h = max(1, src.height//scale)
        out_w = max(1, src.width//scale)
        a = src.read(1, out_shape=(out_h,out_w),
                     resampling=Resampling.nearest)
        return np.unique(a).tolist()

In [ ]:
# ============================================================
# 17. FIND NATIVE FCS30 TILES BY TRUE SPATIAL INTERSECTION
# ============================================================
FCS30_SITE_TILES = {}

for site in COMPARE_SITES:
    ref = site_context_file(site)
    rows = []

    for p in FCS30_NATIVE_CANDIDATES:
        try:
            ov = overlap_fraction(p, ref)
            if ov > 0:
                m = metadata(p)
                rows.append({
                    "site": site,
                    "file": p.name,
                    "path": str(p),
                    "xres": m["xres"],
                    "yres": m["yres"],
                    "crs": str(m["crs"]),
                    "overlap_fraction": ov,
                    "anisotropy": pixel_anisotropy(m["xres"],m["yres"])
                })
        except Exception:
            pass

    tab = pd.DataFrame(rows)
    if not tab.empty:
        tab = tab.sort_values("overlap_fraction", ascending=False)

    print(f"\n=== {site}: intersecting native-source FCS30 tiles ===")
    display(tab)

    FCS30_SITE_TILES[site] = [
        Path(p) for p in tab["path"].tolist()
    ] if not tab.empty else []

    print("Number of intersecting tiles:", len(FCS30_SITE_TILES[site]))

In [ ]:
# ============================================================
# 18. BOUNDED FCS30 MOSAIC ON THE NATIVE SOURCE GRID
# ============================================================
# Merge ONLY the native FCS30 pixels covering the SiDroForest study area.
# This avoids allocating the full combined extent of the source tiles.

FCS30_MOSAICS = {}
FCS30_MOSAIC_DIR = OUT / "native_fcs30_mosaics"
FCS30_MOSAIC_DIR.mkdir(parents=True, exist_ok=True)

def bounded_native_mosaic(site, tiles, reference_path, out_path):
    if not tiles:
        return None

    srcs = [rasterio.open(p) for p in tiles]
    try:
        first = srcs[0]
        if first.crs is None:
            raise ValueError("FCS30 source raster has no CRS.")

        for src in srcs[1:]:
            if src.crs != first.crs:
                raise ValueError(f"FCS30 tiles for {site} do not share one CRS.")

        with rasterio.open(reference_path) as ref:
            if ref.crs is None:
                raise ValueError("SiDroForest reference raster has no CRS.")
            left, bottom, right, top = transform_bounds(
                ref.crs, first.crs, *ref.bounds, densify_pts=21
            )

        xres = abs(first.transform.a)
        yres = abs(first.transform.e)

        # One native FCS30 pixel margin.
        merge_bounds = (
            left - xres, bottom - yres,
            right + xres, top + yres
        )

        print(
            f"{site}: bounded FCS30 merge",
            f"| source pixel = {xres} x {yres}",
            f"| bounds = {tuple(round(v, 5) for v in merge_bounds)}"
        )

        mosaic, transform = merge(
            srcs,
            bounds=merge_bounds,
            res=(xres, yres),
            nodata=first.nodata,
            method="first",
            target_aligned_pixels=True
        )

        profile = first.profile.copy()
        profile.update(
            driver="GTiff",
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=transform,
            count=1,
            compress="deflate",
            tiled=True,
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(mosaic[0], 1)

        return out_path
    finally:
        for src in srcs:
            src.close()

for site, tiles in FCS30_SITE_TILES.items():
    if not tiles:
        print(site, ": no native FCS30 tiles found")
        FCS30_MOSAICS[site] = None
        continue

    ref = site_context_file(site)
    out_path = FCS30_MOSAIC_DIR / f"{site}_FCS30_native_bounded.tif"

    try:
        result = bounded_native_mosaic(site, tiles, ref, out_path)
        FCS30_MOSAICS[site] = result

        with rasterio.open(result) as chk:
            print(
                site, "FCS30 bounded mosaic:",
                f"{chk.width} x {chk.height} pixels",
                "| pixel =", abs(chk.transform.a), "x", abs(chk.transform.e),
                "| CRS =", chk.crs,
                "| classes =", unique_values_sample(result)
            )
    except Exception as e:
        FCS30_MOSAICS[site] = None
        print(f"FAILED {site} bounded FCS30 mosaic:", repr(e))

In [ ]:
# ============================================================
# 19. CGLS PROVENANCE AUDIT — LK = LK.tif + LK2.tif
# ============================================================
# Lake Khamra (LK) was covered by TWO LK-specific CGLS files that were
# merged in the original workflow. Do not substitute files from other sites.
#
# This cell:
#   1) inspects LK.tif and LK2.tif only;
#   2) aligns each to the historical merged LK derivative for provenance;
#   3) tests simple horizontal/vertical seams in both component orders;
#   4) saves the best reconstructed LK composite on the historical grid.
#
# The seam reconstruction is a provenance diagnostic. The final scale-aware
# comparison uses the historical merged LK derivative aggregated back to
# nominal 100 m support.

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

# HIST_CGLS_ROOT is configured in the setup cell above.

LK_PARTS = [
    CGLS_ROOT / "LK.tif",
    CGLS_ROOT / "LK2.tif",
]

YA_CGLS_SOURCE = CGLS_ROOT / "YA.tif"

LK_HIST = HIST_CGLS_ROOT / "LK_Cropped_Image_Downsampled.tif"
YA_HIST = HIST_CGLS_ROOT / "YA_Cropped_Image_Downsampled.tif"

LK_RECON_DIR = OUT / "lk_cgls_reconstruction"
LK_RECON_DIR.mkdir(parents=True, exist_ok=True)

LK_RECONSTRUCTED_HIST_GRID = (
    LK_RECON_DIR / "LK_CGLS_reconstructed_historical_grid.tif"
)

VALID_CGLS_RAW = np.array([1, 2, 3, 4], dtype=np.uint8)


def read_historical(path, fallback_crs):
    with rasterio.open(path) as src:
        arr = src.read(1)
        profile = src.profile.copy()

    if profile.get("crs") is None:
        profile["crs"] = fallback_crs

    return arr, profile


def align_candidate_to_profile(candidate_path, dst_profile):
    """Reproject a raw CGLS candidate to a diagnostic reference grid."""
    dst = np.full(
        (dst_profile["height"], dst_profile["width"]),
        255,
        dtype=np.uint8
    )

    with rasterio.open(candidate_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_profile["transform"],
            dst_crs=dst_profile["crs"],
            resampling=Resampling.nearest,
            src_nodata=src.nodata,
            dst_nodata=255
        )

    return dst


def agreement_against_reference(ref, candidate):
    ref_valid = np.isin(ref, VALID_CGLS_RAW)
    cand_valid = np.isin(candidate, VALID_CGLS_RAW)
    common = ref_valid & cand_valid

    n_ref = int(ref_valid.sum())
    n_common = int(common.sum())
    n_correct = int(((ref == candidate) & common).sum())

    return {
        "reference_pixels": n_ref,
        "covered_pixels": n_common,
        "coverage": n_common / n_ref if n_ref else np.nan,
        "correct_pixels": n_correct,
        "agreement_on_covered": n_correct / n_common if n_common else np.nan,
        "end_to_end_reproduction": n_correct / n_ref if n_ref else np.nan,
    }


def best_simple_seam(ref, a, b, name_a="LK.tif", name_b="LK2.tif"):
    """
    Find the best simple vertical or horizontal seam between two aligned
    LK components. Both component orders are tested.

    Score = correctly reproduced raw classes / all valid historical pixels.
    """
    ref_valid = np.isin(ref, VALID_CGLS_RAW)
    total_ref = int(ref_valid.sum())

    if total_ref == 0:
        raise ValueError("Historical LK raster has no valid 1/2/3/4 pixels.")

    valid_a = np.isin(a, VALID_CGLS_RAW)
    valid_b = np.isin(b, VALID_CGLS_RAW)

    corr_a = (a == ref) & ref_valid & valid_a
    corr_b = (b == ref) & ref_valid & valid_b
    cov_a = ref_valid & valid_a
    cov_b = ref_valid & valid_b

    tests = []

    def vertical(left, right, lc, rc, lv, rv, left_name, right_name):
        if ref.shape[1] < 2:
            return
        l_corr = np.cumsum(lc.sum(axis=0))
        r_corr = np.cumsum(rc.sum(axis=0)[::-1])[::-1]
        l_cov = np.cumsum(lv.sum(axis=0))
        r_cov = np.cumsum(rv.sum(axis=0)[::-1])[::-1]

        scores = (l_corr[:-1] + r_corr[1:]) / total_ref
        coverage = (l_cov[:-1] + r_cov[1:]) / total_ref
        i = int(np.nanargmax(scores))
        seam = i + 1

        comp = np.full(ref.shape, 255, dtype=np.uint8)
        comp[:, :seam] = left[:, :seam]
        comp[:, seam:] = right[:, seam:]

        tests.append({
            "orientation": "vertical",
            "first": left_name,
            "second": right_name,
            "seam_index": seam,
            "reproduction_score": float(scores[i]),
            "coverage": float(coverage[i]),
            "composite": comp,
        })

    def horizontal(top, bottom, tc, bc, tv, bv, top_name, bottom_name):
        if ref.shape[0] < 2:
            return
        t_corr = np.cumsum(tc.sum(axis=1))
        b_corr = np.cumsum(bc.sum(axis=1)[::-1])[::-1]
        t_cov = np.cumsum(tv.sum(axis=1))
        b_cov = np.cumsum(bv.sum(axis=1)[::-1])[::-1]

        scores = (t_corr[:-1] + b_corr[1:]) / total_ref
        coverage = (t_cov[:-1] + b_cov[1:]) / total_ref
        i = int(np.nanargmax(scores))
        seam = i + 1

        comp = np.full(ref.shape, 255, dtype=np.uint8)
        comp[:seam, :] = top[:seam, :]
        comp[seam:, :] = bottom[seam:, :]

        tests.append({
            "orientation": "horizontal",
            "first": top_name,
            "second": bottom_name,
            "seam_index": seam,
            "reproduction_score": float(scores[i]),
            "coverage": float(coverage[i]),
            "composite": comp,
        })

    vertical(a, b, corr_a, corr_b, cov_a, cov_b, name_a, name_b)
    vertical(b, a, corr_b, corr_a, cov_b, cov_a, name_b, name_a)
    horizontal(a, b, corr_a, corr_b, cov_a, cov_b, name_a, name_b)
    horizontal(b, a, corr_b, corr_a, cov_b, cov_a, name_b, name_a)

    best = max(tests, key=lambda x: x["reproduction_score"])
    return best, tests


# ------------------------------------------------------------
# LK: two-component audit
# ------------------------------------------------------------

with rasterio.open(site_context_file("LK")) as src:
    LK_SITE_CRS = src.crs

print("\n" + "=" * 80)
print("LK CGLS TWO-PART AUDIT")
print("Current LK SiDroForest CRS:", LK_SITE_CRS)
print("Historical merged derivative:", LK_HIST)

if not LK_HIST.exists():
    raise FileNotFoundError(f"Missing historical LK derivative: {LK_HIST}")

for p in LK_PARTS:
    if not p.exists():
        raise FileNotFoundError(f"Missing LK CGLS component: {p}")

lk_hist, lk_hist_profile = read_historical(LK_HIST, LK_SITE_CRS)

print(
    "Historical LK shape:", lk_hist.shape,
    "| diagnostic CRS:", lk_hist_profile["crs"],
    "| resolution:",
    (abs(lk_hist_profile["transform"].a),
     abs(lk_hist_profile["transform"].e)),
    "| classes:", np.unique(lk_hist).tolist()
)

aligned_parts = {}
component_rows = []

for p in LK_PARTS:
    with rasterio.open(p) as src:
        print(
            "\nSOURCE:", p.name,
            "| shape:", (src.height, src.width),
            "| CRS:", src.crs,
            "| resolution:",
            (abs(src.transform.a), abs(src.transform.e)),
            "| bounds:", src.bounds,
            "| classes:", np.unique(src.read(1)).tolist()
        )

    aligned = align_candidate_to_profile(p, lk_hist_profile)
    aligned_parts[p.name] = aligned

    component_rows.append({
        "file": p.name,
        **agreement_against_reference(lk_hist, aligned)
    })

LK_COMPONENT_AUDIT = pd.DataFrame(component_rows)

print("\nIndividual LK component reproduction:")
display(LK_COMPONENT_AUDIT)

best_seam, seam_tests = best_simple_seam(
    lk_hist,
    aligned_parts["LK.tif"],
    aligned_parts["LK2.tif"],
    "LK.tif",
    "LK2.tif"
)

LK_SEAM_AUDIT = pd.DataFrame([
    {k: v for k, v in test.items() if k != "composite"}
    for test in seam_tests
]).sort_values("reproduction_score", ascending=False)

print("\nSimple two-part seam tests:")
display(LK_SEAM_AUDIT)

print("\nBEST SIMPLE LK RECONSTRUCTION")
print("orientation:", best_seam["orientation"])
print("first:", best_seam["first"])
print("second:", best_seam["second"])
print("seam index:", best_seam["seam_index"])
print("coverage:", f"{best_seam['coverage']:.4%}")
print(
    "end-to-end historical reproduction:",
    f"{best_seam['reproduction_score']:.4%}"
)

lk_reconstructed = best_seam["composite"]

save_profile = lk_hist_profile.copy()
save_profile.update(
    driver="GTiff",
    dtype="uint8",
    count=1,
    nodata=255,
    compress="deflate"
)

with rasterio.open(LK_RECONSTRUCTED_HIST_GRID, "w", **save_profile) as dst:
    dst.write(lk_reconstructed.astype(np.uint8), 1)
    dst.update_tags(
        purpose="provenance diagnostic only",
        components="LK.tif + LK2.tif",
        seam_orientation=best_seam["orientation"],
        seam_index=str(best_seam["seam_index"]),
        first_component=best_seam["first"],
        second_component=best_seam["second"],
        historical_reproduction=f"{best_seam['reproduction_score']:.8f}"
    )

print("Saved LK reconstruction diagnostic:", LK_RECONSTRUCTED_HIST_GRID)

# ------------------------------------------------------------
# YA: verify YA only, no cross-site substitution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("YA CGLS AUDIT")

YA_HISTORICAL_AGREEMENT = np.nan

if YA_CGLS_SOURCE.exists() and YA_HIST.exists():
    with rasterio.open(site_context_file("YA")) as src:
        ya_site_crs = src.crs

    ya_hist, ya_hist_profile = read_historical(YA_HIST, ya_site_crs)
    ya_aligned = align_candidate_to_profile(YA_CGLS_SOURCE, ya_hist_profile)
    ya_stats = agreement_against_reference(ya_hist, ya_aligned)
    YA_HISTORICAL_AGREEMENT = ya_stats["agreement_on_covered"]

    print("YA source:", YA_CGLS_SOURCE)
    print("Coverage:", f"{ya_stats['coverage']:.4%}")
    print(
        "Historical agreement on covered pixels:",
        f"{YA_HISTORICAL_AGREEMENT:.4%}"
    )
else:
    print("YA source or historical derivative missing.")


## CGLS reconstruction checkpoint — Lake Khamra

Lake Khamra is treated differently because its original CGLS coverage was assembled from **two LK-specific inputs (`LK.tif` and `LK2.tif`)**. No raster belonging to another study site is used.

The cell above aligns both surviving LK components to the historical merged LK derivative and tests simple horizontal and vertical seams in both component orders. This is a **provenance diagnostic** only.

For the final scale-aware comparison, the historical merged LK derivative is used as the authoritative record of that original merge and is aggregated by categorical majority back to **nominal 100 m support**. This avoids treating the old 10 m resampling as additional spatial information. The LK result should therefore be described as a *nominal-100 m reconstruction from the historical merged CGLS derivative*, not as a recovered native CGLS grid.


In [ ]:
# ============================================================
# 20. PREPARE CGLS COMPARISON GRIDS
# ============================================================
# LK:
#   historical merged CGLS derivative -> categorical MODE aggregation
#   to a square 100 m grid in the LK SiDroForest CRS.
#
# YA:
#   YA.tif directly, because it is the verified square-pixel YA source.
#
# The LK grid is a NOMINAL-support reconstruction and is not claimed
# to be the original native CGLS grid.

from rasterio.transform import from_origin
import math

CGLS_PREP_DIR = OUT / "prepared_cgls_comparison"
CGLS_PREP_DIR.mkdir(parents=True, exist_ok=True)

LK_CGLS_100M = (
    CGLS_PREP_DIR /
    "LK_CGLS_historical_merge_nominal100m.tif"
)


def aggregate_historical_to_square_grid(
    source_path,
    fallback_crs,
    out_path,
    target_resolution=100.0
):
    with rasterio.open(source_path) as src:
        src_arr = src.read(1)
        src_transform = src.transform
        src_crs = src.crs or fallback_crs
        src_bounds = src.bounds

    src_u8 = np.full(src_arr.shape, 255, dtype=np.uint8)
    valid = np.isin(src_arr, VALID_CGLS_RAW)
    src_u8[valid] = src_arr[valid].astype(np.uint8)

    left = math.floor(src_bounds.left / target_resolution) * target_resolution
    right = math.ceil(src_bounds.right / target_resolution) * target_resolution
    bottom = math.floor(src_bounds.bottom / target_resolution) * target_resolution
    top = math.ceil(src_bounds.top / target_resolution) * target_resolution

    width = int(round((right - left) / target_resolution))
    height = int(round((top - bottom) / target_resolution))

    dst_transform = from_origin(
        left, top, target_resolution, target_resolution
    )

    dst = np.full((height, width), 255, dtype=np.uint8)

    reproject(
        source=src_u8,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=src_crs,
        resampling=Resampling.mode,
        src_nodata=255,
        dst_nodata=255
    )

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "dtype": "uint8",
        "crs": src_crs,
        "transform": dst_transform,
        "nodata": 255,
        "compress": "deflate"
    }

    with rasterio.open(out_path, "w", **profile) as dst_file:
        dst_file.write(dst, 1)
        dst_file.update_tags(
            source_type="historical merged CGLS derivative",
            source_components="LK.tif + LK2.tif",
            scale_matching="categorical mode aggregation",
            target_support="nominal 100 m square grid",
            warning=(
                "Reconstructed comparison support; not claimed to be "
                "the recovered native CGLS grid."
            )
        )

    return out_path


with rasterio.open(site_context_file("LK")) as src:
    lk_site_crs = src.crs

LK_CGLS_100M = aggregate_historical_to_square_grid(
    LK_HIST,
    fallback_crs=lk_site_crs,
    out_path=LK_CGLS_100M,
    target_resolution=100.0
)

with rasterio.open(LK_CGLS_100M) as chk:
    print(
        "LK prepared CGLS:",
        LK_CGLS_100M,
        "| shape:", (chk.height, chk.width),
        "| CRS:", chk.crs,
        "| pixel:", (abs(chk.transform.a), abs(chk.transform.e)),
        "| classes:", np.unique(chk.read(1)).tolist()
    )

CGLS_SELECTED = {
    "LK": LK_CGLS_100M,
    "YA": YA_CGLS_SOURCE if YA_CGLS_SOURCE.exists() else None,
}

CGLS_SELECTION_STATUS = {
    "LK": "historical_LK_merge_aggregated_to_nominal_100m",
    "YA": (
        "verified_YA_square_grid"
        if YA_CGLS_SOURCE.exists()
        else "missing"
    ),
}

CGLS_EXCLUSION_REASON = {
    "LK": None,
    "YA": None if YA_CGLS_SOURCE.exists() else "YA.tif not found.",
}

print("\nFINAL CGLS COMPARISON SOURCES")
for site, p in CGLS_SELECTED.items():
    print(site, "->", p)
    print("   status:", CGLS_SELECTION_STATUS[site])


# Product class harmonization

Do not infer class semantics from the numeric values alone.

Target SiDroForest comparison codes:
- `0` = nodata/non-forest/outside comparison
- `1` = evergreen
- `2` = summergreen
- `3` = mixed

Fill the dictionaries below from the **actual FCS30 and CGLS class legends or from the code that produced the six-class derivatives**. Until these mappings are filled, the final comparison intentionally does not run.

In [ ]:
# ============================================================
# 21. CLASS HARMONIZATION RECOVERED FROM SUPPLIED MATERIAL
# ============================================================
# Target convention used in THIS notebook:
# 0 = excluded/non-forest/unknown
# 1 = evergreen needleleaf
# 2 = summergreen needleleaf
# 3 = mixed forest
#
# CGLS source logic recovered from the supplied thesis/manuscript:
# original CGLS 113/123 -> summergreen
# original CGLS 111/121 -> evergreen
# original CGLS 115/125 -> mixed
# original CGLS 116/126 -> unknown forest, analysed separately.
#
# The supplied already-harmonized CGLS derivative uses:
# 1 = summergreen, 2 = evergreen, 3 = mixed, 4 = unknown forest.
# Therefore convert to this notebook's 1=evergreen / 2=summergreen convention:
CGLS_CLASS_MAP = {
    1: 2,  # summergreen -> target 2
    2: 1,  # evergreen   -> target 1
    3: 3,  # mixed       -> target 3
    4: 0,  # unknown forest: excluded from direct class agreement
    255: 0
}

# Native GLC_FCS30 forest classes.
# The previous supplied processing produced harmonized FCS30 rasters with
# 0/1/2/3 classes. Here we perform the equivalent harmonization directly
# on the native product so that the external raster is never upsampled.
#
# Native forest codes used by the supplied FCS30 workflow:
# 71/72 = evergreen needleleaf (open/closed)
# 81/82 = deciduous/summergreen needleleaf (open/closed)
# 91/92 = mixed leaf forest (open/closed)
# All other native classes are excluded from the three-class comparison.
FCS30_CLASS_MAP = {
    71: 1, 72: 1,
    81: 2, 82: 2,
    91: 3, 92: 3
}

def harmonize_classes(arr, mapping):
    out = np.zeros(arr.shape, dtype="uint8")
    for raw, target in mapping.items():
        out[arr == raw] = target
    return out

print("CGLS mapping:", CGLS_CLASS_MAP)
print("FCS30 mapping:", FCS30_CLASS_MAP)

In [ ]:
# ============================================================
# 21B. SANITY CHECK AGAINST SUPPLIED HARMONIZED FCS30 RASTERS
# ============================================================
# The old 10 m FCS30 derivatives are NOT used for the final comparison.
# They are used only as an audit check that the native-code harmonization
# reproduces the same class semantics.

OLD_FCS30_HARMONIZED = {
    "LK": SIDRO / "FCS30_forest_classes/LK_forest_types.tif",
    "YA": SIDRO / "FCS30_forest_classes/YA_forest_types.tif",
}

for site in COMPARE_SITES:
    native = FCS30_MOSAICS.get(site)
    old = OLD_FCS30_HARMONIZED[site]

    if native is None or not Path(native).exists() or not old.exists():
        print(site, ": mapping audit skipped (file unavailable)")
        continue

    with rasterio.open(native) as src:
        raw = src.read(1)
        native_h = harmonize_classes(raw, FCS30_CLASS_MAP)
        native_profile = src.profile.copy()

    # Align the native harmonized map to the old derivative ONLY for this
    # diagnostic. The final analysis below remains on the native product grid.
    with rasterio.open(old) as ref:
        native_on_old = np.zeros((ref.height, ref.width), dtype="uint8")
        reproject(
            source=native_h,
            destination=native_on_old,
            src_transform=native_profile["transform"],
            src_crs=native_profile["crs"],
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            resampling=Resampling.nearest,
            src_nodata=0,
            dst_nodata=0
        )
        old_arr = ref.read(1)

    # Determine whether the old derivative uses the same target convention
    # or the historical 1=summergreen,2=evergreen convention.
    valid = np.isin(native_on_old,[1,2,3]) & np.isin(old_arr,[1,2,3])
    if valid.sum() == 0:
        print(site, ": no common forest pixels for mapping audit")
        continue

    same = np.mean(native_on_old[valid] == old_arr[valid])
    swapped = np.mean(
        native_on_old[valid] ==
        np.where(old_arr[valid]==1, 2,
                 np.where(old_arr[valid]==2, 1, old_arr[valid]))
    )

    print(
        site,
        "| mapping audit n =", int(valid.sum()),
        "| agreement same convention =", round(float(same), 3),
        "| agreement if 1/2 swapped =", round(float(swapped), 3)
    )

In [ ]:
# ============================================================
# 22. AGGREGATE SIDROFOREST TO AN EXISTING PRODUCT GRID
# ============================================================
def aggregate_sidro_to_product_grid(sidro_arr, sidro_profile, product_path):
    with rasterio.open(product_path) as target:
        if target.crs is None or sidro_profile.get("crs") is None:
            raise ValueError("Both rasters require valid CRS metadata.")

        dst = np.zeros((target.height,target.width), dtype="uint8")

        reproject(
            source=sidro_arr.astype("uint8"),
            destination=dst,
            src_transform=sidro_profile["transform"],
            src_crs=sidro_profile["crs"],
            dst_transform=target.transform,
            dst_crs=target.crs,
            resampling=Resampling.mode,
            src_nodata=0,
            dst_nodata=0
        )
        product = target.read(1)
        profile = target.profile.copy()

    return dst, product, profile

def comparison_metrics(a,b):
    valid = np.isin(a,[1,2,3]) & np.isin(b,[1,2,3])
    if valid.sum() == 0:
        return None

    av = a[valid]
    bv = b[valid]
    out = {
        "n_valid": int(valid.sum()),
        "overall_agreement": float(np.mean(av == bv))
    }

    for cls,name in [(1,"evergreen"),(2,"summergreen"),(3,"mixed")]:
        a_pos = av == cls
        b_pos = bv == cls
        tp = np.sum(a_pos & b_pos)
        fp = np.sum(a_pos & ~b_pos)
        fn = np.sum(~a_pos & b_pos)
        union = np.sum(a_pos | b_pos)

        # SiDroForest is treated as the map being compared; these are
        # agreement metrics, not ground-truth accuracy metrics.
        out[f"{name}_sidro_pct"] = np.mean(a_pos)*100
        out[f"{name}_product_pct"] = np.mean(b_pos)*100
        out[f"{name}_precision_agreement"] = tp/(tp+fp) if (tp+fp) else np.nan
        out[f"{name}_recall_agreement"] = tp/(tp+fn) if (tp+fn) else np.nan
        p = out[f"{name}_precision_agreement"]
        r = out[f"{name}_recall_agreement"]
        out[f"{name}_f1_agreement"] = (
            2*p*r/(p+r) if np.isfinite(p) and np.isfinite(r) and (p+r) else np.nan
        )
        out[f"{name}_iou"] = tp/union if union else np.nan
    return out

In [ ]:
# ============================================================
# 23. FINAL SCALE-AWARE COMPARISON — v15
# ============================================================
# FCS30:
#   bounded native-source mosaics.
#
# CGLS YA:
#   verified square-pixel YA source.
#
# CGLS LK:
#   historical LK merge (LK.tif + LK2.tif) aggregated back to
#   nominal 100 m square support.
#
# In all comparisons, the finer SiDroForest contextual map is reduced
# to the product grid by categorical MODE / majority class.

import math
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.warp import transform_bounds, reproject, Resampling

PRODUCT_FILES = {
    "LK": {
        "FCS30": FCS30_MOSAICS.get("LK"),
        "CGLS_LC100": CGLS_SELECTED.get("LK"),
    },
    "YA": {
        "FCS30": FCS30_MOSAICS.get("YA"),
        "CGLS_LC100": CGLS_SELECTED.get("YA"),
    },
}

PRODUCT_MAPS = {
    "FCS30": FCS30_CLASS_MAP,
    "CGLS_LC100": CGLS_CLASS_MAP,
}


def bounded_product_read(product_path, target_profile, pad_pixels=1):
    target_crs = target_profile["crs"]
    t = target_profile["transform"]
    w = target_profile["width"]
    h = target_profile["height"]

    left = t.c
    top = t.f
    right = left + t.a * w
    bottom = top + t.e * h

    xmin, xmax = sorted([left, right])
    ymin, ymax = sorted([bottom, top])

    with rasterio.open(product_path) as src:
        if src.crs is None:
            raise ValueError(f"{product_path} has no CRS")

        pb = transform_bounds(
            target_crs, src.crs,
            xmin, ymin, xmax, ymax,
            densify_pts=21
        )

        clip_left = max(pb[0], src.bounds.left)
        clip_bottom = max(pb[1], src.bounds.bottom)
        clip_right = min(pb[2], src.bounds.right)
        clip_top = min(pb[3], src.bounds.top)

        if clip_left >= clip_right or clip_bottom >= clip_top:
            raise ValueError("No spatial overlap.")

        win = from_bounds(
            clip_left, clip_bottom, clip_right, clip_top,
            transform=src.transform
        )

        col0 = max(0, math.floor(win.col_off) - pad_pixels)
        row0 = max(0, math.floor(win.row_off) - pad_pixels)
        col1 = min(src.width, math.ceil(win.col_off + win.width) + pad_pixels)
        row1 = min(src.height, math.ceil(win.row_off + win.height) + pad_pixels)

        win = Window(
            col_off=col0,
            row_off=row0,
            width=col1-col0,
            height=row1-row0
        )

        arr = src.read(1, window=win)
        transform = src.window_transform(win)

        profile = src.profile.copy()
        profile.update(
            height=arr.shape[0],
            width=arr.shape[1],
            transform=transform
        )

    return arr, profile


def aggregate_sidro_to_product_grid(
    sidro,
    sidro_profile,
    product_arr,
    product_profile
):
    dst = np.zeros(product_arr.shape, dtype=np.uint8)

    reproject(
        source=sidro.astype(np.uint8),
        destination=dst,
        src_transform=sidro_profile["transform"],
        src_crs=sidro_profile["crs"],
        dst_transform=product_profile["transform"],
        dst_crs=product_profile["crs"],
        resampling=Resampling.mode,
        src_nodata=0,
        dst_nodata=0
    )

    return dst


comparison_rows = []

for site, products in PRODUCT_FILES.items():
    print("\n" + "=" * 80)
    print("SITE:", site)

    sidro_path = site_context_file(site)

    with rasterio.open(sidro_path) as src:
        sidro = src.read(1)
        sidro_profile = src.profile.copy()

    print("SiDroForest:", sidro_path)
    print(
        "grid:", sidro.shape,
        "| CRS:", sidro_profile["crs"],
        "| classes:", np.unique(sidro).tolist()
    )

    for product_name, product_path in products.items():
        print(f"\n--- {site} / {product_name} ---")
        print("Product:", product_path)

        if product_path is None:
            print(
                "SKIP:",
                CGLS_EXCLUSION_REASON.get(site)
                if product_name == "CGLS_LC100"
                else "No product source."
            )
            continue

        try:
            product, product_profile = bounded_product_read(
                product_path,
                sidro_profile,
                pad_pixels=1
            )

            print(
                "Bounded product grid:", product.shape,
                "| CRS:", product_profile["crs"],
                "| pixel:",
                (
                    abs(product_profile["transform"].a),
                    abs(product_profile["transform"].e)
                ),
                "| raw classes:", np.unique(product).tolist()
            )

            print("Aggregating SiDroForest by MODE / majority class ...")

            sidro_agg = aggregate_sidro_to_product_grid(
                sidro,
                sidro_profile,
                product,
                product_profile
            )

            product_h = harmonize_classes(
                product,
                PRODUCT_MAPS[product_name]
            )

            sidro_forest = np.isin(sidro_agg, [1, 2, 3])
            product_forest = np.isin(product_h, [1, 2, 3])
            common = sidro_forest & product_forest

            n_sidro = int(sidro_forest.sum())
            n_product = int(product_forest.sum())
            n_common = int(common.sum())

            print("SiDroForest forest cells:", n_sidro)
            print("Product forest cells:", n_product)
            print("Common forest cells:", n_common)

            if n_common == 0:
                print("NO COMPARISON: zero common forest support.")
                continue

            n_exact = int((sidro_agg[common] == product_h[common]).sum())
            exact = n_exact / n_common

            print(
                "Exact class matches:",
                f"{n_exact} / {n_common} = {exact:.4f}"
            )

            metrics = comparison_metrics(
                sidro_agg[common],
                product_h[common]
            )

            status = (
                CGLS_SELECTION_STATUS.get(site)
                if product_name == "CGLS_LC100"
                else "native_FCS30_source"
            )

            row = {
                "site": site,
                "product": product_name,
                "n_valid": n_common,
                "overall_agreement": exact,
                "product_height": product.shape[0],
                "product_width": product.shape[1],
                "product_crs": str(product_profile["crs"]),
                "product_xres": abs(product_profile["transform"].a),
                "product_yres": abs(product_profile["transform"].e),
                "resampling_method": "mode_majority",
                "source_file": Path(product_path).name,
                "source_status": status,
            }

            if isinstance(metrics, dict):
                row.update(metrics)
            elif isinstance(metrics, pd.DataFrame) and len(metrics):
                row.update(metrics.iloc[0].to_dict())

            comparison_rows.append(row)

            print(
                f"SUCCESS: {site} {product_name} | "
                f"n_valid={n_common} | agreement={exact:.4f} | "
                f"status={status}"
            )

        except Exception as e:
            print("FAILED:", type(e).__name__, "-", str(e))


scale_aware_comparison = pd.DataFrame(comparison_rows)

print("\n" + "=" * 80)
print("FINAL SCALE-AWARE PRODUCT COMPARISON")

if not scale_aware_comparison.empty:
    display(scale_aware_comparison)

    output_csv = OUT / "scale_aware_global_product_comparison.csv"
    scale_aware_comparison.to_csv(output_csv, index=False)
    print("Saved:", output_csv)
else:
    print("No successful comparisons were produced.")


In [ ]:
# ============================================================
# 24. EXPORT AUDIT INFORMATION — v15
# ============================================================

audit_xlsx = OUT / "global_product_scale_aware_audit_v15.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="openpyxl") as writer:

    if "LK_COMPONENT_AUDIT" in globals():
        LK_COMPONENT_AUDIT.to_excel(
            writer, sheet_name="LK_components", index=False
        )

    if "LK_SEAM_AUDIT" in globals():
        LK_SEAM_AUDIT.to_excel(
            writer, sheet_name="LK_seam_tests", index=False
        )

    for site, tiles in FCS30_SITE_TILES.items():
        rows = []

        for p in tiles:
            try:
                m = metadata(p)
                rows.append({
                    "site": site,
                    "file": p.name,
                    "path": str(p),
                    "crs": str(m["crs"]),
                    "xres": m["xres"],
                    "yres": m["yres"],
                    "anisotropy": pixel_anisotropy(
                        m["xres"], m["yres"]
                    ),
                    "overlap_fraction": overlap_fraction(
                        p, site_context_file(site)
                    ),
                })
            except Exception as e:
                rows.append({
                    "site": site,
                    "file": getattr(p, "name", str(p)),
                    "path": str(p),
                    "error": str(e),
                })

        fcs_tab = pd.DataFrame(rows)

        if not fcs_tab.empty:
            fcs_tab.to_excel(
                writer,
                sheet_name=f"{site}_FCS30_tiles"[:31],
                index=False
            )

    if (
        "scale_aware_comparison" in globals()
        and isinstance(scale_aware_comparison, pd.DataFrame)
        and not scale_aware_comparison.empty
    ):
        scale_aware_comparison.to_excel(
            writer, sheet_name="comparison", index=False
        )

    source_rows = [
        {
            "site": "LK",
            "source": str(LK_CGLS_100M),
            "status": CGLS_SELECTION_STATUS["LK"],
            "underlying_components": "LK.tif + LK2.tif",
            "final_support": "nominal 100 m",
            "note": (
                "Historical merged derivative aggregated by categorical "
                "mode; not claimed as recovered native CGLS grid."
            ),
        },
        {
            "site": "YA",
            "source": str(CGLS_SELECTED.get("YA")),
            "status": CGLS_SELECTION_STATUS["YA"],
            "underlying_components": "YA.tif",
            "final_support": "existing square product grid",
            "note": "Verified against historical YA derivative.",
        },
    ]

    pd.DataFrame(source_rows).to_excel(
        writer, sheet_name="CGLS_sources", index=False
    )

print("Saved audit workbook:", audit_xlsx)


# Part H — publication-ready figures for the revised manuscript

This section creates a compact figure set designed to **replace redundant figures rather than add more pages**.

### Suggested main figures generated here

1. **Method workflow** — separates previous RF work from the new contextual/generalisation analysis.
2. **Independent evaluation** — shows classified versus masked outcomes transparently.
3. **Final hard-label forest-composition maps** — Lake Khamra and Yakutsk.
4. **Window × threshold sensitivity** — generated earlier in Part C.
5. **Scale-matched class composition** — explains global-product disagreement at matched spatial support.

### Suggested supplementary figures

- hard-label parameter stability;
- probability-based entropy / uncertainty;
- hard-label versus probability-method parameter sensitivity.

The existing study-area figure can be retained if it is already clear and publication quality.


In [ ]:
# ============================================================
# 25. MAIN FIGURE — ANALYSIS WORKFLOW
# ============================================================
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 3.6))
ax.set_xlim(0, 13)
ax.set_ylim(0, 4)
ax.axis("off")

boxes = [
    (0.3, 1.3, 2.0, 1.35, "Sentinel-2\nfeatures", "Input"),
    (2.8, 1.3, 2.1, 1.35, "Existing\nSiDroForest RF", "Previous work"),
    (5.4, 1.3, 2.2, 1.35, "Hard RF labels\n+ RF confidence", "Pixel output"),
    (8.1, 1.3, 2.3, 1.35, "Spatial composition\n100–1000 m", "New contextual analysis"),
    (10.9, 1.3, 1.8, 1.35, "Evergreen\nSummergreen\nMixed", "Composition map"),
]

for x, y, w, h, label, small in boxes:
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.05,rounding_size=0.08",
        linewidth=1.2,
        facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(
        x + w/2, y + h/2 + 0.10,
        label,
        ha="center", va="center",
        fontsize=10
    )
    ax.text(
        x + w/2, y + 0.12,
        small,
        ha="center", va="bottom",
        fontsize=7
    )

for i in range(len(boxes) - 1):
    x1 = boxes[i][0] + boxes[i][2]
    y1 = boxes[i][1] + boxes[i][3] / 2
    x2 = boxes[i+1][0]
    y2 = boxes[i+1][1] + boxes[i+1][3] / 2

    ax.add_patch(FancyArrowPatch(
        (x1 + 0.08, y1),
        (x2 - 0.08, y2),
        arrowstyle="->",
        mutation_scale=13,
        linewidth=1.2
    ))

ax.text(
    6.9, 0.45,
    "Independent observations → RF evaluation and abstention coverage     |     "
    "Window × threshold sensitivity     |     Scale-matched FCS30 / CGLS agreement",
    ha="center", va="center",
    fontsize=9
)

fig.suptitle("SiDroForest analysis framework", fontsize=13)

png = MANUSCRIPT_FIG_DIR / "Fig_workflow.png"
pdf = MANUSCRIPT_FIG_DIR / "Fig_workflow.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


In [ ]:
# ============================================================
# 26. MAIN FIGURE — INDEPENDENT EVALUATION WITH ABSTENTIONS
# ============================================================
def extended_confusion_table(audit_df):
    rows = ["evergreen", "summergreen"]
    cols = ["evergreen", "summergreen", "masked"]

    out = pd.DataFrame(0, index=rows, columns=cols, dtype=int)

    if audit_df is None or audit_df.empty:
        return out

    for r in rows:
        for c in cols:
            out.loc[r, c] = int(
                np.sum(
                    (audit_df["_reference"] == r)
                    & (audit_df["_prediction"] == c)
                )
            )

    return out


field_mat = extended_confusion_table(
    field2021_audit if "field2021_audit" in globals() else None
)
pang_mat = extended_confusion_table(
    pangaea_audit if "pangaea_audit" in globals() else None
)

fig, axes = plt.subplots(
    1, 2, figsize=(10.5, 4.5),
    constrained_layout=True
)

vmax = max(1, field_mat.values.max(), pang_mat.values.max())

for ax, mat, title in [
    (axes[0], field_mat, "2021 independent field polygons"),
    (axes[1], pang_mat, "Independent SiDroForest/PANGAEA locations"),
]:
    ax.imshow(mat.values, vmin=0, vmax=vmax)

    ax.set_xticks(range(3))
    ax.set_xticklabels(
        ["Evergreen", "Summergreen", "Masked"],
        rotation=25,
        ha="right"
    )
    ax.set_yticks(range(2))
    ax.set_yticklabels(["Evergreen", "Summergreen"])

    ax.set_xlabel("Model outcome")
    ax.set_ylabel("Reference")
    ax.set_title(title, fontsize=10)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(
                j, i, str(mat.iloc[i, j]),
                ha="center", va="center",
                fontsize=11
            )

fig.suptitle(
    "Independent evaluation including masked / abstained outcomes",
    fontsize=12
)

png = MANUSCRIPT_FIG_DIR / "Fig_independent_validation_abstention.png"
pdf = MANUSCRIPT_FIG_DIR / "Fig_independent_validation_abstention.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


In [ ]:
# ============================================================
# 27. MAIN FIGURE — FINAL HARD-LABEL FOREST-COMPOSITION MAPS
# ============================================================
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

class_cmap = ListedColormap([
    "#D9D9D9",
    "#1B5E20",
    "#D9A400",
    "#7B4FA3",
])

legend_handles = [
    Patch(facecolor="#1B5E20", label="Evergreen"),
    Patch(facecolor="#D9A400", label="Summergreen"),
    Patch(facecolor="#7B4FA3", label="Mixed"),
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5.5),
    constrained_layout=True
)

for ax, site, title in [
    (axes[0], "LK", "Lake Khamra"),
    (axes[1], "YA", "Yakutsk"),
]:
    path = site_context_file(site)

    with rasterio.open(path) as src:
        arr = src.read(1)

    masked = np.ma.masked_where(arr == 0, arr)

    ax.imshow(
        masked,
        cmap=class_cmap,
        vmin=0,
        vmax=3,
        interpolation="nearest"
    )
    ax.set_title(title)
    ax.axis("off")

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02)
)

fig.suptitle(
    f"Final hard-label forest composition "
    f"({ADOPTED_SCALE_M} m neighbourhood, "
    f"{ADOPTED_THRESHOLD:.2f} dominance threshold)",
    fontsize=12
)

png = MANUSCRIPT_FIG_DIR / "Fig_final_hard_label_maps.png"
pdf = MANUSCRIPT_FIG_DIR / "Fig_final_hard_label_maps.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


In [ ]:
# ============================================================
# 28. MAIN FIGURE — SCALE-MATCHED CLASS COMPOSITION
# ============================================================
if "scale_aware_comparison" not in globals():
    raise RuntimeError(
        "Run the final scale-aware comparison in Part G first."
    )

class_cols = [
    ("evergreen", "#1B5E20"),
    ("summergreen", "#D9A400"),
    ("mixed", "#7B4FA3"),
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 5.2),
    constrained_layout=True
)

for ax, site in zip(axes, ["LK", "YA"]):
    sub = scale_aware_comparison[
        scale_aware_comparison["site"] == site
    ].copy()

    labels = []
    rows = []

    for _, r in sub.iterrows():
        product = r["product"]

        labels.append(f"SiDroForest\n@ {product} grid")
        rows.append([
            r["evergreen_sidro_pct"],
            r["summergreen_sidro_pct"],
            r["mixed_sidro_pct"],
        ])

        labels.append(product)
        rows.append([
            r["evergreen_product_pct"],
            r["summergreen_product_pct"],
            r["mixed_product_pct"],
        ])

    vals = np.asarray(rows, dtype=float)
    x = np.arange(len(labels))
    bottom = np.zeros(len(labels))

    for idx, (name, color) in enumerate(class_cols):
        ax.bar(
            x,
            vals[:, idx],
            bottom=bottom,
            label=name.capitalize(),
            color=color
        )
        bottom += vals[:, idx]

    ax.set_ylim(0, 100)
    ax.set_ylabel("Forest-class proportion (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        rotation=25,
        ha="right",
        fontsize=8
    )
    ax.set_title("Lake Khamra" if site == "LK" else "Yakutsk")

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels_leg,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.04)
)

fig.suptitle(
    "Forest-class composition at matched product support",
    fontsize=12
)

png = MANUSCRIPT_FIG_DIR / "Fig_scale_matched_class_composition.png"
pdf = MANUSCRIPT_FIG_DIR / "Fig_scale_matched_class_composition.pdf"

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


In [ ]:
# ============================================================
# 29. SUPPLEMENTARY FIGURE — HARD vs PROBABILITY SENSITIVITY
# ============================================================
sens = stability_df[
    stability_df["site"].isin(["LK", "YA"])
].copy()

plot_labels = []
values = []

for site in ["LK", "YA"]:
    for method in ["hard_label", "probability"]:
        hit = sens[
            (sens["site"] == site)
            & (sens["method"] == method)
        ]

        if hit.empty:
            continue

        plot_labels.append(
            f"{site}\n"
            + ("Hard labels" if method == "hard_label"
               else "Probabilities")
        )
        values.append(
            float(
                hit.iloc[0][
                    "parameter_sensitive_0p2_to_0p8_pct"
                ]
            )
        )

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(np.arange(len(values)), values)
ax.set_xticks(np.arange(len(values)))
ax.set_xticklabels(plot_labels)
ax.set_ylabel("Parameter-sensitive forest pixels (%)")
ax.set_ylim(0, max(100, max(values) * 1.1 if values else 100))
ax.set_title(
    "Sensitivity to neighbourhood and dominance-threshold choices"
)

png = (
    MANUSCRIPT_FIG_DIR
    / "FigS_hard_vs_probability_parameter_sensitivity.png"
)
pdf = (
    MANUSCRIPT_FIG_DIR
    / "FigS_hard_vs_probability_parameter_sensitivity.pdf"
)

fig.savefig(png, dpi=400, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", pdf)


In [ ]:
# ============================================================
# 30. FIGURE INVENTORY + DRAFT CAPTIONS
# ============================================================
figure_inventory = pd.DataFrame([
    {
        "role": "Main",
        "file": "Fig_workflow.pdf",
        "suggested_use": "Methods overview",
        "draft_caption": (
            "Overview of the revised SiDroForest analysis framework. "
            "The previously developed Random Forest provides pixel-level "
            "evergreen and summergreen classifications. The present study "
            "evaluates independent generalisation, derives spatial forest "
            "composition from hard RF labels across neighbourhood scales, "
            "tests sensitivity to neighbourhood width and dominance "
            "threshold, and compares the resulting maps with global "
            "land-cover products at matched spatial support."
        )
    },
    {
        "role": "Main",
        "file": "Fig_independent_validation_abstention.pdf",
        "suggested_use": "Independent evaluation results",
        "draft_caption": (
            "Independent evaluation including masked or abstained outcomes. "
            "Rows show reference forest type and columns show the model "
            "outcome. Masked observations are reported explicitly rather "
            "than removed from the evaluation denominator."
        )
    },
    {
        "role": "Main",
        "file": "Fig_final_hard_label_maps.pdf",
        "suggested_use": "Final forest-composition maps",
        "draft_caption": (
            f"Final hard-label contextual forest-composition maps for "
            f"Lake Khamra and Yakutsk using a {ADOPTED_SCALE_M} m "
            f"neighbourhood and {ADOPTED_THRESHOLD:.2f} dominance "
            f"threshold. A neighbourhood is assigned evergreen or "
            f"summergreen when that type reaches the dominance threshold; "
            f"otherwise it is classified as mixed."
        )
    },
    {
        "role": "Main",
        "file": "Fig_sensitivity_hard_label.pdf",
        "suggested_use": "Parameter sensitivity",
        "draft_caption": (
            "Sensitivity of mixed-forest extent to neighbourhood width and "
            "dominance threshold for the hard-label contextual method. "
            "Values are the percentage of forest pixels classified as mixed. "
            "The outlined cell marks the nominal 500 m / 0.75 manuscript "
            "setting."
        )
    },
    {
        "role": "Main",
        "file": "Fig_scale_matched_class_composition.pdf",
        "suggested_use": "Global-product comparison",
        "draft_caption": (
            "Forest-class composition of SiDroForest and the global "
            "land-cover products at matched spatial support. SiDroForest "
            "was aggregated by categorical majority to each comparison "
            "grid. The figure shows product agreement and compositional "
            "differences; the global products are not treated as ground truth."
        )
    },
    {
        "role": "Supplement",
        "file": "FigS_hard_label_parameter_stability.pdf",
        "suggested_use": "Parameter stability",
        "draft_caption": (
            "Spatial stability of hard-label mixed-forest classification "
            "across all tested neighbourhood widths and dominance thresholds."
        )
    },
    {
        "role": "Supplement",
        "file": "FigS_probability_entropy.pdf",
        "suggested_use": "RF probability uncertainty",
        "draft_caption": (
            "Probability-based compositional entropy at 500 m context, "
            "retained as a continuous uncertainty and compositional-ambiguity "
            "measure rather than the primary categorical mixed-forest definition."
        )
    },
    {
        "role": "Supplement",
        "file": "FigS_hard_vs_probability_parameter_sensitivity.pdf",
        "suggested_use": "Method sensitivity comparison",
        "draft_caption": (
            "Percentage of forest pixels with intermediate mixed-class "
            "stability across the tested parameter grid for hard-label and "
            "probability-based contextual formulations."
        )
    },
])

display(figure_inventory)

inventory_csv = (
    MANUSCRIPT_FIG_DIR
    / "figure_inventory_and_caption_drafts.csv"
)
figure_inventory.to_csv(inventory_csv, index=False)

print("Saved:", inventory_csv)
print("Figure directory:", MANUSCRIPT_FIG_DIR)
